
# 12 — Data Imputation and Encoding

**Scope:** The missing and incorrect values are handled, categorical features are encoded.


Since we decided to predict missing and incorrect values, we have to split the training data into training and validation sets before applying imputation and encoding.

# Table of Contents

Take this as an example for a Table of Contents for your notebook.
We have to fix all the names and sections according to what we actually do in the notebook.

<a class="anchor" id="top"></a>

** **

1. [Setup & Imports](#sec-setup)

2. [Load Data & Initial Audit](#sec-load)

3. [Split the Training data into training and validation sets](#2sec-Exploratory-Data-Analysis)
    
   2.1 [Incoherencies](#2.1-Incoherencies) <br>
   
   &emsp; 2.1.1 [Address Identified Incoherencies](#2.1.1-Address-Identified-Incoherencies) <br>
    
4. [Data Cleaning & Preprocessing](#3.-Data-Cleaning-&-Preprocessing)

   3.1 [Duplicates](#3.1-Duplicates) <br>
    
   3.2 [Feature Engineering](#3.2-Feature-Engineering) <br>
   
   &emsp; 3.2.1 [Data Type Conversions](#3.2.1-Data-Type-Conversions) <br>
   
   &emsp; 3.2.2 [Encoding](#3.2.2-Encoding) <br>
   
   &emsp; 3.2.3 [Other Transformations](#3.2.3-Other-Transformations) <br>
    
   &emsp; 3.2.4 [Unique Feature-Pair Analysis](#3.2.4-Unique-Feature-Pair-Analysis) <br> 

   3.3 [Train-Test Split](#3.3-Train-Test-Split) <br>
   
   3.4 [Missing Values](#3.4-Missing-Values) <br>
    
   3.5 [Outliers](#3.5-Outliers) <br>

   3.6 [Visualisations](#3.6-Visualisations) <br><br>
   

<a id="sec-setup"></a>
## 1. Setup & Imports

In [1]:
import os, re, math, warnings
from pathlib import Path
from datetime import datetime
import json
import pandas as pd
import numpy as np
import re
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import clone, BaseEstimator, TransformerMixin

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
pd.set_option("mode.copy_on_write", True)
warnings.filterwarnings("ignore")

RANDOM_STATE = 42  # for reproducibility of any sampling


<a id="sec-load"></a>
## 2. Load Data


In [2]:
# Load the data paths
data_dir = "../data/"

# Load the raw data into a pandas dataframe
df = pd.read_csv(os.path.join(data_dir, "processed_data/11_processed_train_data.csv"))
X_test = pd.read_csv(os.path.join(data_dir, "processed_data/11_processed_test_data.csv"))

# put carID as Index
df.set_index("carID", inplace=True)
X_test.set_index("carID", inplace=True)

print("Loaded shape:", df.shape)
display(df.head(3))

print("Loaded test shape:", X_test.shape)
display(X_test.head(3))


Loaded shape: (74467, 13)


,Brand,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
carID,,,,,,,,,,,,,
69512,Volkswagen,Golf,2016.0,22290.0,Semi-Auto,28421.0,Petrol,NaN,11.417268,2.0,63.0,4.0,0.0
53000,Toyota,Yaris,2019.0,13790.0,Manual,4589.0,Petrol,145.0,47.900000,1.5,50.0,1.0,0.0
6366,Audi,Q2,2019.0,24990.0,Semi-Auto,3624.0,Petrol,145.0,40.900000,1.5,56.0,4.0,0.0


Loaded test shape: (32567, 12)


,Brand,model,year,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
carID,,,,,,,,,,,,
89856,Hyundai,i30,NaN,Automatic,30700.0,Petrol,205.0,41.5,1.6,61.0,3.0,0.0
106581,Volkswagen,Tiguan,2017.0,Semi-Auto,NaN,Petrol,150.0,38.2,2.0,60.0,2.0,0.0
80886,BMW,2 Series,2016.0,Automatic,36792.0,Petrol,125.0,51.4,1.5,94.0,2.0,0.0


### Split the Training data into training and validation sets

In [3]:
# Split df into Training and Validation Set (60/20/20 total)

# Separate features and target from the training data
X = df.drop(columns=["price"])
y = df["price"]

# Split df (80% of total data) into training and validation
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.25,        # 25% of 80% train = 20% of total
    random_state=42,
)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_val shape:   {X_val.shape}")
print(f"y_val shape:   {y_val.shape}")
print(f"X_test shape:  {X_test.shape}")


X_train shape: (55850, 12)
y_train shape: (55850,)
X_val shape:   (18617, 12)
y_val shape:   (18617,)
X_test shape:  (32567, 12)


# Handling Missing Values

- Numerical: fill with the median
- Categorical: fill with the most frequent value

In [4]:
def missing_report(X: pd.DataFrame, name: str) -> pd.DataFrame:
    mv = X.isna().sum()
    mv = mv[mv > 0].sort_values(ascending=False)
    if mv.empty:
        print(f"[{name}] No missing values found. (n_rows={len(X)})")
        return pd.DataFrame(columns=["n_missing", "pct_missing"])
    pct = (mv / len(X) * 100).round(2)
    report = pd.DataFrame({"n_missing": mv, "pct_missing": pct})
    print(f"[{name}] Missing Values (n_rows={len(X)}):")
    display(report)
    return report

# Reports for the splits
mv_train = missing_report(X_train, "X_train")
mv_val   = missing_report(X_val,   "X_val")
mv_test  = missing_report(X_test,  "X_test")

# Helpful for the next step (imputation/encoding):
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print(f"#numeric_cols: {len(numeric_cols)} → {numeric_cols[:10]}{' ...' if len(numeric_cols) > 10 else ''}")
print(f"#categorical_cols: {len(categorical_cols)} → {categorical_cols[:10]}{' ...' if len(categorical_cols) > 10 else ''}")


[X_train] Missing Values (n_rows=55850):


,n_missing,pct_missing
mpg,6750,12.09
tax,6079,10.88
transmission,1666,2.98
engineSize,1548,2.77
previousOwners,1425,2.55
mileage,1380,2.47
paintQuality%,1378,2.47
model,1228,2.20
hasDamage,1136,2.03
fuelType,1113,1.99


[X_val] Missing Values (n_rows=18617):


,n_missing,pct_missing
mpg,2254,12.11
tax,2029,10.90
transmission,552,2.97
engineSize,507,2.72
paintQuality%,475,2.55
previousOwners,463,2.49
model,425,2.28
mileage,421,2.26
hasDamage,385,2.07
fuelType,366,1.97


[X_test] Missing Values (n_rows=32567):


,n_missing,pct_missing
mpg,3840,11.79
tax,3469,10.65
year,1007,3.09
transmission,968,2.97
engineSize,875,2.69
mileage,859,2.64
paintQuality%,793,2.43
previousOwners,765,2.35
model,734,2.25
fuelType,656,2.01


#numeric_cols: 8 → ['year', 'mileage', 'tax', 'mpg', 'engineSize', 'paintQuality%', 'previousOwners', 'hasDamage']
#categorical_cols: 4 → ['Brand', 'model', 'transmission', 'fuelType']


# Pipeline Definition

In order to impute the missing values of some of the categorical features we're gonna be using RandomForestClassifier. However, RandomForestClassifer only works with numerical values, so for that matter the other categorical features (whose values are gonna be used by the RFClassifier) need to be encoded into numerical values. In order to do this we implemented a pipeline for each one of these categorical features in order for them to be encoded since most of them have really different cardinalities.

In [5]:
def get_pipeline_feature_names(pipeline, input_columns):
    """
    Given a fitted pipeline (with SafeColumnTransformer and custom imputers),
    return the output feature names as a list.
    """
    # Start with the input columns
    feature_names = list(input_columns)
    
    # Recursively handle nested pipelines / column transformers
    def _get_names(transformer, input_feats):
        names_out = []
        
        if hasattr(transformer, 'transformers_'):  # ColumnTransformer
            for name, trans, cols in transformer.transformers_:
                # Only include columns that exist in input
                cols = [c for c in cols if c in input_feats]
                
                if hasattr(trans, 'get_feature_names_out'):
                    # scikit-learn transformers like OneHotEncoder, OrdinalEncoder
                    names_out.extend(trans.get_feature_names_out(cols))
                elif hasattr(trans, 'transformers_') or hasattr(trans, 'steps'):
                    # nested ColumnTransformer or Pipeline
                    if hasattr(trans, 'transformers_'):
                        # ColumnTransformer
                        nested = _get_names(trans, cols)
                        names_out.extend(nested)
                    else:
                        # Pipeline: apply last step
                        last_step = trans.steps[-1][1]
                        if hasattr(last_step, 'get_feature_names_out'):
                            names_out.extend(last_step.get_feature_names_out(cols))
                        else:
                            names_out.extend(cols)
                else:
                    # custom transformer without get_feature_names_out
                    names_out.extend(cols)
            return names_out
        elif hasattr(transformer, 'steps'):
            # Pipeline
            for step_name, step in transformer.steps:
                input_feats = _get_names(step, input_feats)
            return input_feats
        else:
            # plain transformer (no change in feature count)
            return input_feats

    feature_names = _get_names(pipeline, feature_names)
    return feature_names

In [6]:
class SafeColumnTransformer(ColumnTransformer):
    """ColumnTransformer that ignores missing columns instead of erroring."""

    def _hstack(self, Xs, *args, **kwargs):
        # sklearn internals changed across versions: some call _hstack with extra kwargs
        # Accept any additional arguments/keywords and forward them to the parent implementation.
        return super()._hstack(Xs, *args, **kwargs)

    def _subset_transformers(self, X):
        valid_transformers = []
        for (name, trans, cols) in self.transformers:
            # Identify which columns actually exist
            existing = [c for c in cols if c in X.columns]
            if len(existing) == 0:
                # skip transformer entirely
                continue
            valid_transformers.append((name, trans, existing))
        return valid_transformers

    def fit(self, X, y=None, **fit_params):
        # Temporarily restrict transformers to columns present in X
        original = self.transformers
        self.transformers = self._subset_transformers(X)
        try:
            return super().fit(X, y, **fit_params)
        finally:
            # Always restore original transformers
            self.transformers = original

    def transform(self, X):
        # Temporarily restrict transformers to columns present in X
        original = self.transformers
        self.transformers = self._subset_transformers(X)
        try:
            Xt = super().transform(X)
        finally:
            # Always restore original transformers
            self.transformers = original
        return Xt

    def fit_transform(self, X, y=None, **fit_params):
        # Temporarily restrict transformers to columns present in X
        original = self.transformers
        self.transformers = self._subset_transformers(X)
        try:
            return super().fit_transform(X, y, **fit_params)
        finally:
            # Always restore original transformers
            self.transformers = original

# Impute Missing Categorical values and ML Prediction

## impute_transmission

The method imputes missing transmission types.
It first fills them with the most frequent transmission within each car model.
Remaining gaps are predicted using a Random Forest classifier trained on features such as brand, model, fuel type, engine size, year, mpg, and tax.

In [7]:

class TransmissionImputer(BaseEstimator, TransformerMixin):
    def __init__(self, pipeline_template, min_model_count=10):
        self.pipeline_template = pipeline_template
        self.min_model_count = min_model_count

    def fit(self, X, y=None):
        df = X.copy()

        # store lookup tables learned from training data
        counts = df.dropna(subset=['transmission'])['model'].value_counts()
        self.valid_models_ = counts[counts >= self.min_model_count].index

        self.model_modes_ = (
            df.dropna(subset=['model', 'transmission'])
              .groupby('model')['transmission']
              .agg(lambda s: s.mode().iloc[0])
        )

        # Train ML model for remaining missing values
        train_df = df[df['transmission'].notna()].copy()

        features = [
            "Brand", "model", "fuelType",
            "engineSize", "year", "mpg", "tax",
        ]

        pipe = clone(self.pipeline_template)
        pipe.fit(train_df[features], train_df['transmission'])

        self.ml_model_ = pipe
        self.features_ = features
        return self

    def transform(self, X):
        df = X.copy()

        # rule-based fill
        for model in self.valid_models_:
            mask = (df['model'] == model) & (df['transmission'].isna())
            if model in self.model_modes_:
                df.loc[mask, 'transmission'] = self.model_modes_[model]

        # ML fallback
        missing_mask = df['transmission'].isna()
        if missing_mask.any():
            df.loc[missing_mask, 'transmission'] = self.ml_model_.predict(
                df.loc[missing_mask, self.features_]
            )
        return df

## impute_fuelType

The method imputes missing fuel types.
It first fills them with the per-model mode for models that have at least min_model_count rows.
For remaining gaps it trains a Random Forest classifier on brand, model, transmission, engine size, year, and mpg.

In [8]:
class FuelTypeImputer(BaseEstimator, TransformerMixin):
    def __init__(self, pipeline_template, min_model_count=10):
        self.pipeline_template = pipeline_template
        self.min_model_count = min_model_count

    def fit(self, X, y=None):
        df = X.copy()

        counts = df.dropna(subset=['fuelType'])['model'].value_counts()
        self.valid_models_ = counts[counts >= self.min_model_count].index

        self.model_modes_ = (
            df.dropna(subset=['model', 'fuelType'])
              .groupby('model')['fuelType']
              .agg(lambda s: s.mode().iloc[0])
        )

        features = ['Brand', 'model', 'transmission',
                    'engineSize', 'year', 'mpg']
        train_df = df[df['fuelType'].notna()]

        pipe = clone(self.pipeline_template)
        pipe.fit(train_df[features], train_df['fuelType'])

        self.ml_model_ = pipe
        self.features_ = features
        return self

    def transform(self, X):
        df = X.copy()

        # rule-based fill
        for model in self.valid_models_:
            mask = (df['model'] == model) & (df['fuelType'].isna())
            df.loc[mask, 'fuelType'] = self.model_modes_.get(model, np.nan)

        # ML fallback
        missing = df['fuelType'].isna()
        if missing.any():
            df.loc[missing, 'fuelType'] = self.ml_model_.predict(
                df.loc[missing, self.features_]
            )
        return df

## impute_mileage

The method imputes missing mileage values.
It replaces them with the median mileage of cars from the same production year.
If some values are still missing, they are filled with the overall median mileage.

In [9]:
class MileageImputer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        df = X.copy()

        self.year_medians_ = df.groupby('year')['mileage'].median()
        self.global_median_ = df['mileage'].median()
        return self

    def transform(self, X):
        df = X.copy()

        df['mileage'] = df['mileage'].fillna(
            df['year'].map(self.year_medians_)
        )
        df['mileage'] = df['mileage'].fillna(self.global_median_)
        return df

## impute_year

The method imputes missing production years.
It assigns each car to a mileage bin and fills missing years with the median year of that bin, assuming cars with similar mileage have similar ages.
Remaining gaps are filled with the overall median year.

In [10]:
class YearImputer(BaseEstimator, TransformerMixin):
    def __init__(self, bins=30):
        self.bins = bins

    def fit(self, X, y=None):
        df = X.copy()

        df['mileage_bin'] = pd.cut(df['mileage'], bins=self.bins)
        self.bin_medians_ = df.groupby('mileage_bin')['year'].median()
        self.global_median_ = df['year'].median()
        return self

    def transform(self, X):
        df = X.copy()

        df['mileage_bin'] = pd.cut(df['mileage'], bins=self.bins)

        df['year'] = df['year'].fillna(
            df['mileage_bin'].map(self.bin_medians_)
        )
        df['year'] = df['year'].fillna(self.global_median_)

        df.drop(columns='mileage_bin', inplace=True)
        return df

## impute_mpg

It first fills values using the median mpg for the same model and fuel type.
If unavailable, it falls back to brand and fuel type, then to fuel type only, and finally to the overall median, ensuring consistent and realistic mpg values.

In [11]:
class MPGImputer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        df = X.copy()

        self.model_medians_ = df.groupby(['model', 'fuelType'])['mpg'].median()
        self.brand_medians_ = df.groupby(['Brand', 'fuelType'])['mpg'].median()
        self.fuel_medians_ = df.groupby(['fuelType'])['mpg'].median()
        self.global_median_ = df['mpg'].median()
        return self

    def transform(self, X):
        df = X.copy()

        for idx, row in df[df['mpg'].isna()].iterrows():
            km = (row['model'], row['fuelType'])
            kb = (row['Brand'], row['fuelType'])
            if km in self.model_medians_.index:
                df.at[idx, 'mpg'] = self.model_medians_.loc[km]
            elif kb in self.brand_medians_.index:
                df.at[idx, 'mpg'] = self.brand_medians_.loc[kb]

        df['mpg'] = df['mpg'].fillna(df['fuelType'].map(self.fuel_medians_))
        df['mpg'] = df['mpg'].fillna(self.global_median_)
        return df

## impute_paintQuality

The method imputes missing paint quality values.
It simply replaces all missing entries in paintQuality% with the column’s median, providing a stable and unbiased estimate for missing data.

In [12]:
class PaintQualityImputer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.median_ = X['paintQuality%'].median()
        return self

    def transform(self, X):
        df = X.copy()
        df['paintQuality%'] = df['paintQuality%'].fillna(self.median_)
        return df

## impute_previousOwners

The method imputes missing values for previousOwners.
It fills all missing entries with the median number of previous owners, ensuring consistency while avoiding the influence of outliers.

In [13]:
class PreviousOwnersImputer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.median_ = X['previousOwners'].median()
        return self

    def transform(self, X):
        df = X.copy()
        df['previousOwners'] = df['previousOwners'].fillna(self.median_)
        return df

## impute_tax

The method imputes missing tax values.
It first fills them using the median tax of cars with the same model, fuel type, and mpg range.
If that information is missing, it falls back to brand and fuel type, then to fuel type only, and finally to the global median, ensuring realistic tax estimates based on car efficiency and type.

In [14]:
class TaxImputer(BaseEstimator, TransformerMixin):
    def __init__(self, bins=15):
        self.bins = bins

    def fit(self, X, y=None):
        df = X.copy()

        df['mpg_bin'] = pd.cut(df['mpg'], bins=self.bins)

        self.medians_one_ = df.groupby(['model', 'fuelType', 'mpg_bin'])['tax'].median()
        self.medians_two_ = df.groupby(['Brand', 'fuelType', 'mpg_bin'])['tax'].median()
        self.medians_three_ = df.groupby(['fuelType', 'mpg_bin'])['tax'].median()
        self.fuel_medians_ = df.groupby('fuelType')['tax'].median()
        self.global_median_ = df['tax'].median()
        return self

    def transform(self, X):
        df = X.copy()

        df['mpg_bin'] = pd.cut(df['mpg'], bins=self.bins)

        for idx, row in df[df['tax'].isna()].iterrows():
            km = (row['model'], row['fuelType'], row['mpg_bin'])
            kb = (row['Brand'], row['fuelType'], row['mpg_bin'])
            ke = (row['fuelType'], row['mpg_bin'])

            if km in self.medians_one_.index:
                df.at[idx, 'tax'] = self.medians_one_.loc[km]
            elif kb in self.medians_two_.index:
                df.at[idx, 'tax'] = self.medians_two_.loc[kb]
            elif ke in self.medians_three_.index:
                df.at[idx, 'tax'] = self.medians_three_.loc[ke]

        df['tax'] = df['tax'].fillna(df['fuelType'].map(self.fuel_medians_))
        df['tax'] = df['tax'].fillna(self.global_median_)
        return df

## impute_engineSize

The method imputes missing engine sizes.
It fills them with the median engine size for the same model and fuel type.
If that is unavailable, it uses brand and fuel type, then the model median, and finally the overall median, ensuring realistic and consistent engine size estimates.

In [15]:
class EngineSizeImputer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        df = X.copy()

        self.model_medians_ = df.groupby(['model', 'fuelType'])['engineSize'].median()
        self.brand_medians_ = df.groupby(['Brand', 'fuelType'])['engineSize'].median()
        self.fuel_medians_ = df.groupby(['model'])['engineSize'].median()
        self.global_median_ = df['engineSize'].median()
        return self

    def transform(self, X):
        df = X.copy()

        for idx, row in df[df['engineSize'].isna()].iterrows():
            km = (row['model'], row['fuelType'])
            kb = (row['Brand'], row['fuelType'])
            if km in self.model_medians_.index:
                df.at[idx, 'engineSize'] = self.model_medians_.loc[km]
            elif kb in self.brand_medians_.index:
                df.at[idx, 'engineSize'] = self.brand_medians_.loc[kb]

        df['engineSize'] = df['engineSize'].fillna(df['model'].map(self.fuel_medians_))
        df['engineSize'] = df['engineSize'].fillna(self.global_median_)
        return df 

## impute_brand

The method imputes missing car brands.
It first fills them using the most common brand for each model, assuming models usually belong to one brand.
If values remain missing, it trains a Random Forest classifier on transmission, engine size, fuel type, and mpg to predict the brand, combining logical mapping with data-driven prediction.

In [16]:
class BrandImputer(BaseEstimator, TransformerMixin):
    def __init__(self, pipeline_template, min_model_count=20):
        self.pipeline_template = pipeline_template
        self.min_model_count = min_model_count

    def fit(self, X, y=None):
        df = X.copy()

        # rule-based brand per model
        self.model_to_brand_ = (
            df.dropna(subset=['Brand', 'model'])
              .groupby('model')['Brand']
              .agg(lambda s: s.mode().iloc[0])
        )

        # ML fallback
        features = ['transmission', 'engineSize',
                    'fuelType', 'mpg', 'model']
        train_df = df[df['Brand'].notna()]

        pipe = clone(self.pipeline_template)
        pipe.fit(train_df[features], train_df['Brand'])

        self.ml_model_ = pipe
        self.features_ = features
        return self

    def transform(self, X):
        df = X.copy()

        # rule-based fill
        mask = df['Brand'].isna() & df['model'].notna()
        df.loc[mask, 'Brand'] = df.loc[mask, 'model'].map(self.model_to_brand_)

        # ML fallback
        missing = df['Brand'].isna()
        if missing.any():
            df.loc[missing, 'Brand'] = self.ml_model_.predict(
                df.loc[missing, self.features_]
            )
        return df

## impute_model

The method imputes missing car models.
It first fills them with the most common model for each brand and transmission combination.
If values remain missing, it trains a Random Forest classifier using features such as brand, year, engine size, mpg, tax, mileage, fuel type, and transmission to predict the correct model.

In [17]:
class ModelImputer(BaseEstimator, TransformerMixin):
    def __init__(self, pipeline_template, min_brand_count=20):
        self.pipeline_template = pipeline_template
        self.min_brand_count = min_brand_count

    def fit(self, X, y=None):
        df = X.copy()

        # rule-based: most frequent model per (Brand, transmission)
        self.lookup_ = (
            df.dropna(subset=['Brand', 'transmission', 'model'])
              .groupby(['Brand', 'transmission'])['model']
              .agg(lambda s: s.value_counts().idxmax())
        )

        # ML fallback
        features = ['Brand', 'year', 'engineSize', 'mpg',
                    'tax', 'mileage', 'fuelType', 'transmission']
        train_df = df[df['model'].notna()]

        pipe = clone(self.pipeline_template)
        pipe.fit(train_df[features], train_df['model'])

        self.ml_model_ = pipe
        self.features_ = features
        return self

    def transform(self, X):
        df = X.copy()

        # rule-based fill
        for idx, row in df[df['model'].isna()].iterrows():
            key = (row['Brand'], row['transmission'])
            if key in self.lookup_.index:
                df.at[idx, 'model'] = self.lookup_.loc[key]

        # ML fallback
        missing = df['model'].isna()
        if missing.any():
            df.loc[missing, 'model'] = self.ml_model_.predict(
                df.loc[missing, self.features_]
            )
        return df

In [18]:
# Used for low-cardinality categorical features
low_cardinality_pipeline = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))  # or OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
])

# Used for high-cardinality categorical features like model
high_cardinality_pipeline = Pipeline([
    ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ])

# SafeColumnTransformer is basically a ColumnTransformer that ignores missing columns instead of erroring.
encoder = SafeColumnTransformer([
    ('model_ord', high_cardinality_pipeline, ['model']),
    ('brand_ohe', low_cardinality_pipeline, ['Brand']),
    ('fuel_ohe', low_cardinality_pipeline, ['fuelType']),
    ('transmission_ohe', low_cardinality_pipeline, ['transmission'])
], remainder='passthrough')

imputation_template = Pipeline([
    ('encoder', encoder),
    ("rf_model", RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42))
])

rule_imputer = Pipeline([
        ("transmission_imp", TransmissionImputer(imputation_template)),
        ("fuelType_imp", FuelTypeImputer(imputation_template)),
        ("brand_imp", BrandImputer(imputation_template)),
        ("model_imp", ModelImputer(imputation_template)),
        ("mileage_imp", MileageImputer()),
        ("year_imp", YearImputer()),
        ("mpg_imp", MPGImputer()),
        ("paint_imp", PaintQualityImputer()),
        ("previousOwners_imp", PreviousOwnersImputer()),
        ("tax_imp", TaxImputer()),
        ("engineSize_imp", EngineSizeImputer()),
    ])


full_pipeline = Pipeline([
    ("rule_based", rule_imputer)
])

## Impute Missing Feature Values in Training Data

In [19]:
%%time
X_train["hasDamage"] = X_train["hasDamage"].fillna(1)
full_pipeline.fit(X_train, y_train)
X_train_transformed = full_pipeline.transform(X_train)

# get feature names from pipeline (may be shorter/longer than actual transformed columns)
feature_names = get_pipeline_feature_names(full_pipeline, X_train.columns)

X_train = pd.DataFrame(X_train_transformed, columns=feature_names)
print("Transformed X_train shape:", X_train.shape)
X_train.head()

Transformed X_train shape: (55850, 12)
CPU times: user 28.2 s, sys: 365 ms, total: 28.6 s
Wall time: 28.8 s


,Brand,model,year,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
carID,,,,,,,,,,,,
60965,Opel,Corsa,2017.0,Manual,18000.0,Petrol,150.0,55.4,1.4,76.0,1.0,0.0
22100,Ford,Focus,2019.0,Manual,6220.0,Petrol,145.0,58.9,1.0,81.0,6.0,0.0
884,Audi,Q7,2020.0,Automatic,1403.5,Diesel,145.0,33.6,3.0,35.0,4.0,0.0
10083,BMW,2 Series,2019.0,Semi-Auto,7315.0,Petrol,145.0,50.4,2.0,86.0,4.0,0.0
13929,BMW,X3,2015.0,Semi-Auto,58779.0,Diesel,145.0,54.3,2.0,74.0,3.0,0.0


In [20]:
# quick check
# Look at the Missing values
missing_values = X_train.isnull().sum()

print("Missing values in train:")
print(missing_values[missing_values > 0]) 
print("\n")

Missing values in train:
Series([], dtype: int64)




## Impute Missing Feature Values in Validation Data

### Impute Missing Transmission Values in Validation Set

In [21]:
%%time
X_val["hasDamage"] = X_val["hasDamage"].fillna(1)
full_pipeline.fit(X_train, y_train)
X_val_transformed = full_pipeline.transform(X_val)

# get feature names from pipeline (may be shorter/longer than actual transformed columns)
feature_names = get_pipeline_feature_names(full_pipeline, X_val.columns)

X_val = pd.DataFrame(X_val_transformed, columns=feature_names)
print("Transformed X_val shape:", X_val.shape)
X_val.head()

Transformed X_val shape: (18617, 12)
CPU times: user 20.1 s, sys: 264 ms, total: 20.3 s
Wall time: 19.4 s


,Brand,model,year,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
carID,,,,,,,,,,,,
38667,Mercedes-Benz,GLC-Class,2019.0,Semi-Auto,2566.0,Diesel,145.0,44.8,2.0,98.0,0.0,0.0
55200,Toyota,Aygo,2018.0,Semi-Auto,25148.0,Petrol,150.0,67.3,1.0,75.0,0.0,0.0
59907,Opel,Zafira,2017.0,Manual,28657.0,Petrol,200.0,42.2,1.4,51.0,4.0,0.0
59388,Opel,Astra,2017.0,Manual,22330.5,Petrol,145.0,51.4,1.4,45.0,4.0,0.0
46722,Škoda,Yeti Outdoor,2017.0,Manual,38881.0,Petrol,125.0,51.4,1.2,92.0,4.0,0.0


In [22]:
# quick check
# Look at the Missing values
missing_values = X_train.isnull().sum()

print("Missing values in train:")
print(missing_values[missing_values > 0]) 
print("\n")
missing_values = X_val.isnull().sum()
print("Missing values in validation:")
print(missing_values[missing_values > 0]) 

Missing values in train:
Series([], dtype: int64)


Missing values in validation:
Series([], dtype: int64)


## Impute Missing Feature Values in Test Data

In [23]:
%%time
X_test["hasDamage"] = X_test["hasDamage"].fillna(1)
full_pipeline.fit(X_train, y_train)
X_test_transformed = full_pipeline.transform(X_test)
# get feature names from pipeline (may be shorter/longer than actual transformed columns)
feature_names = get_pipeline_feature_names(full_pipeline, X_test.columns)
X_test = pd.DataFrame(X_test_transformed, columns=feature_names)
print("Transformed X_test shape:", X_test.shape)
X_test.head()

Transformed X_test shape: (32567, 12)
CPU times: user 20.9 s, sys: 46.2 ms, total: 21 s
Wall time: 21.6 s


,Brand,model,year,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
carID,,,,,,,,,,,,
89856,Hyundai,i30,2017.0,Automatic,30700.0,Petrol,205.0,41.5,1.6,61.0,3.0,0.0
106581,Volkswagen,Tiguan,2017.0,Semi-Auto,22330.5,Petrol,150.0,38.2,2.0,60.0,2.0,0.0
80886,BMW,2 Series,2016.0,Automatic,36792.0,Petrol,125.0,51.4,1.5,94.0,2.0,0.0
100174,Opel,Grandland X,2019.0,Manual,5533.0,Petrol,145.0,44.1,1.2,77.0,1.0,0.0
81376,BMW,1 Series,2019.0,Semi-Auto,9058.0,Diesel,150.0,51.4,2.0,45.0,4.0,0.0


In [24]:
# quick check
# Look at the Missing values
missing_values = X_test.isnull().sum()

print("Missing values in test:")
print(missing_values[missing_values > 0]) 

Missing values in test:
Series([], dtype: int64)


# Feature Engineering

## Creation of Features in Training Set for Vehicle Efficiency and Usage

This feature set captures key aspects of a car’s efficiency, usage, and ownership.
mpg_diff_transmission measures how a car’s fuel efficiency compares to others with the same transmission.
car_age represents how old the car is, while efficiency_ratio reflects how fuel-efficient the engine is relative to its size.
mileage_per_year shows how intensively the car has been used.
owners_flag marks cars with more than two previous owners, and previousOwners_sq models the non-linear impact of ownership count.
Finally, engine_tax_ratio indicates how efficient the engine is in relation to the car’s tax cost.

In [25]:
trans_medians = X_train.groupby('transmission')['mpg'].median()
X_train['mpg_median_trans'] = X_train['transmission'].map(trans_medians)
X_val['mpg_median_trans'] = X_val['transmission'].map(trans_medians)
X_test['mpg_median_trans'] = X_test['transmission'].map(trans_medians)

In [26]:
X_train['mpg_diff_transmission'] = X_train['mpg'] - X_train['mpg_median_trans']
X_train['mpg_diff_transmission'] = X_train['mpg_diff_transmission'].fillna(0)

X_train = X_train.drop(columns=["mpg_median_trans"])

# age affects cars much more than the year: car_age in years
X_train['car_age'] = 2020 - X_train['year']
# it is important how efficent the motor is
X_train['efficiency_ratio'] = X_train['mpg'] / (X_train['engineSize'] + 0.1)

X_train['mileage_per_year'] = X_train['mileage'] / (X_train['car_age'] + 1)

#brand_popularity = X_train['Brand'].value_counts(normalize=True)
#X_train['brand_popularity'] = X_train['Brand'].map(brand_popularity)

X_train['owners_flag'] = (X_train['previousOwners'] > 2).astype(int)

X_train['previousOwners_sq'] = X_train['previousOwners'] ** 2

X_train['engine_tax_ratio'] = X_train['engineSize'] / (X_train['tax'] + 1)

In [27]:
X_train.describe()

,year,mileage,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage,mpg_diff_transmission,car_age,efficiency_ratio,mileage_per_year,owners_flag,previousOwners_sq,engine_tax_ratio
count,55850.000000,55850.000000,55850.000000,55850.000000,55850.000000,55850.000000,55850.000000,55850.000000,55850.000000,55850.000000,55850.000000,55850.000000,55850.000000,55850.000000,55850.000000
mean,2017.064082,23352.951521,118.075838,54.862284,1.668963,64.249855,2.011961,0.020340,-0.502251,2.935918,34.265217,5360.875040,0.391406,6.078890,0.131386
std,2.141554,21549.519010,65.106758,11.382111,0.554559,20.379646,1.425111,0.141162,11.131651,2.141554,12.823114,3927.994195,0.488069,6.192531,0.396488
min,2000.000000,1.000000,0.000000,9.429179,1.000000,1.638913,0.000000,0.000000,-48.270821,0.000000,2.284061,0.076923,0.000000,0.000000,0.003021
25%,2016.000000,7500.000000,30.000000,47.100000,1.200000,47.000000,1.000000,0.000000,-8.100000,1.000000,25.681818,2834.000000,0.000000,1.000000,0.009589
50%,2017.000000,17494.500000,145.000000,55.400000,1.600000,64.000000,2.000000,0.000000,0.000000,3.000000,31.785714,4860.916667,0.000000,4.000000,0.013245
75%,2019.000000,32475.000000,145.000000,62.800000,2.000000,82.000000,3.000000,0.000000,8.000000,4.000000,43.705882,7050.482143,1.000000,9.000000,0.032258
max,2020.000000,323000.000000,580.000000,94.100000,6.200000,99.000000,6.000000,1.000000,40.800000,20.000000,67.545455,101153.348285,1.000000,36.000000,3.822758


## Creation of Features in Validation Set for Vehicle Efficiency and Usage

In [28]:
X_val['mpg_diff_transmission'] = X_val['mpg'] - X_val['mpg_median_trans']
X_val['mpg_diff_transmission'] = X_val['mpg_diff_transmission'].fillna(0)

X_val = X_val.drop(columns=["mpg_median_trans"])

# age affects cars much more than the year: car_age in years
X_val['car_age'] = 2020 - X_val['year']
# it is important how efficent the motor is, 
X_val['efficiency_ratio'] = X_val['mpg'] / (X_val['engineSize'] + 0.1)

X_val['mileage_per_year'] = X_val['mileage'] / (X_val['car_age'] + 1)

#brand_popularity = X_val['Brand'].value_counts(normalize=True)
#X_val['brand_popularity'] = X_val['Brand'].map(brand_popularity)

X_val['owners_flag'] = (X_val['previousOwners'] > 2).astype(int)

X_val['previousOwners_sq'] = X_val['previousOwners'] ** 2

X_val['engine_tax_ratio'] = X_val['engineSize'] / (X_val['tax'] + 1)

## Creation of Features in Test Set for Vehicle Efficiency and Usage

In [29]:
X_test['mpg_diff_transmission'] = X_test['mpg'] - X_test['mpg_median_trans']
X_test['mpg_diff_transmission'] = X_test['mpg_diff_transmission'].fillna(0)

X_test = X_test.drop(columns=["mpg_median_trans"])

# age affects cars much more than the year: car_age in years
X_test['car_age'] = 2020 - X_test['year']
# it is important how efficent the motor is, 
X_test['efficiency_ratio'] = X_test['mpg'] / (X_test['engineSize'] + 0.1)

X_test['mileage_per_year'] = X_test['mileage'] / (X_test['car_age'] + 1)

#brand_popularity = X_test['Brand'].value_counts(normalize=True)
#X_test['brand_popularity'] = X_test['Brand'].map(brand_popularity)

X_test['owners_flag'] = (X_test['previousOwners'] > 2).astype(int)

X_test['previousOwners_sq'] = X_test['previousOwners'] ** 2

X_test['engine_tax_ratio'] = X_test['engineSize'] / (X_test['tax'] + 1)

# Encoding

In [30]:
def get_feature_names_from_safe_ct(ct: SafeColumnTransformer) -> list[str]:
    """
    Extract feature names from a fitted SafeColumnTransformer.
    
    Works for:
      - OneHotEncoder inside pipelines for low-cardinality features
    
    Returns a flat list of column names matching the transformed output.
    """
    feature_names = []

    for name, transformer, columns in ct.transformers_:
        if transformer == 'passthrough':
            # Keep original column names for numeric / passthrough features
            feature_names.extend(columns)
        elif isinstance(transformer, Pipeline):
            # Assume last step is the actual transformer (usually OneHotEncoder)
            last_step = transformer.steps[-1][1]
            if hasattr(last_step, 'get_feature_names_out'):
                feature_names.extend(last_step.get_feature_names_out(columns))
            else:
                # fallback: just use original column names
                feature_names.extend(columns)
        else:
            # Single transformer (not a pipeline)
            if hasattr(transformer, 'get_feature_names_out'):
                feature_names.extend(transformer.get_feature_names_out(columns))
            else:
                feature_names.extend(columns)

    return feature_names


preprocessor_fitted = clone(encoder)
preprocessor_fitted.fit(X_train)

# Transform datasets
X_train_enc = preprocessor_fitted.transform(X_train)
X_val_enc   = preprocessor_fitted.transform(X_val)
X_test_enc  = preprocessor_fitted.transform(X_test)

# Recover proper column names
all_cols = get_pipeline_feature_names(preprocessor_fitted, X_train.columns)

# Convert to DataFrames
X_train_enc_df = pd.DataFrame(X_train_enc, columns=all_cols, index=X_train.index)
X_val_enc_df   = pd.DataFrame(X_val_enc, columns=all_cols, index=X_val.index)
X_test_enc_df  = pd.DataFrame(X_test_enc, columns=all_cols, index=X_test.index)

In [31]:
# print the columns after encoding
print("Columns after encoding:")
print(X_test_enc_df.columns.tolist())


# Print the number of columns after encoding for test val and train
print("Number of columns after encoding:")
print("X_train_enc:", X_train_enc_df.shape[1])
print("X_val_enc:", X_val_enc_df.shape[1])
print("X_test_enc:", X_test_enc_df.shape[1])

Columns after encoding:
['model', 'Brand_Audi', 'Brand_BMW', 'Brand_Ford', 'Brand_Hyundai', 'Brand_Mercedes-Benz', 'Brand_Opel', 'Brand_Toyota', 'Brand_Volkswagen', 'Brand_Škoda', 'fuelType_Diesel', 'fuelType_Electric', 'fuelType_Hybrid', 'fuelType_Other', 'fuelType_Petrol', 'transmission_Automatic', 'transmission_Manual', 'transmission_Other', 'transmission_Semi-Auto', 'year', 'mileage', 'tax', 'mpg', 'engineSize', 'paintQuality%', 'previousOwners', 'hasDamage', 'mpg_diff_transmission', 'car_age', 'efficiency_ratio', 'mileage_per_year', 'owners_flag', 'previousOwners_sq', 'engine_tax_ratio']
Number of columns after encoding:
X_train_enc: 34
X_val_enc: 34
X_test_enc: 34


# Scaling

After encoding all our values are numerical, but we need to scale the features for better model performance.
The Values for mileage can be very high compared to other features, so scaling is important.



In [32]:
scaler = RobustScaler()

# numeric cols after FE (on train)
num_after_enc = X_train_enc_df.select_dtypes(include=["number"]).columns

# fit on TRAIN only
scaler.fit(X_train_enc_df[num_after_enc])
# copy
x_train_final = X_train_enc_df.copy()
x_val_final   = X_val_enc_df.copy()
x_test_final  = X_test_enc_df.copy()

# scale train and val on same cols
x_train_final[num_after_enc] = scaler.transform(X_train_enc_df[num_after_enc])
x_val_final[num_after_enc]   = scaler.transform(X_val_enc_df[num_after_enc])

# for test: only the intersection of cols
test_cols = [c for c in num_after_enc if c in X_test_enc_df.columns]
x_test_final[test_cols] = scaler.transform(X_test_enc_df[test_cols])

In [33]:
# Check for NaN values in each dataset
def check_nan(df, name):
    nan_cols = df.columns[df.isna().any()].tolist()
    if nan_cols:
        print(f"{name} has NaN values in columns: {nan_cols}")
    else:
        print(f"{name} has no NaN values.")


check_nan(x_train_final, "x_train_final")
check_nan(y_train.to_frame(), "y_train")
check_nan(x_val_final, "x_val_final")
check_nan(y_val.to_frame(), "y_val")
check_nan(x_test_final, "x_test_final")

x_train_final has no NaN values.
y_train has no NaN values.
x_val_final has no NaN values.
y_val has no NaN values.
x_test_final has no NaN values.


# Output Save

In [34]:
# Put CarID index back as a column
x_train_final = x_train_final.reset_index().rename(columns={'index': 'CarID'})
x_val_final = x_val_final.reset_index().rename(columns={'index': 'CarID'})
x_test_final = x_test_final.reset_index().rename(columns={'index': 'CarID'})

In [35]:
# Save Processed Datasets
# Save X_train, y_train, X_val, y_val, and X_test separately.
# This structure is cleaner for later model loading and avoids re-splitting.

output_dir = os.path.join(data_dir, "encoded_data")
os.makedirs(output_dir, exist_ok=True)

# Save feature and target sets separately
x_train_final.to_csv(os.path.join(output_dir, "12_X_train.csv"), index=False)
y_train.to_csv(os.path.join(output_dir, "12_y_train.csv"), index=False)

x_val_final.to_csv(os.path.join(output_dir, "12_X_val.csv"), index=False)
y_val.to_csv(os.path.join(output_dir, "12_y_val.csv"), index=False)

x_test_final.to_csv(os.path.join(output_dir, "12_X_test.csv"), index=False)

print("Processed data saved successfully (X/y separated):")
print(f"X_train: {x_train_final.shape}, y_train: {y_train.shape}")
print(f"X_val:   {x_val_final.shape}, y_val: {y_val.shape}")
print(f"X_test:  {x_test_final.shape}")


Processed data saved successfully (X/y separated):
X_train: (55850, 35), y_train: (55850,)
X_val:   (18617, 35), y_val: (18617,)
X_test:  (32567, 35)
